# Anemia Types Classification (CBC blood tests)

Predict one of 9 diagnoses (healthy + 8 conditions) from 14 complete-blood-count (CBC) values, then wrap the model in a small Gradio demo.

| | |
|---|---|
| **Task** | Multi-class classification (9 imbalanced classes) |
| **Data** | Kaggle `ehababoelnaga/anemia-types-classification`: 1,281 patients, 14 CBC features |
| **Model** | Random Forest (100 trees, `class_weight='balanced'`, `ccp_alpha=0.02`) |
| **Result** | **97.8% test accuracy**, macro F1 0.94 (321 test samples) |

## Approach
- Encoded the 9 diagnosis labels as integers.
- Stratified 75/25 train/test split (`random_state=13`) so rare classes appear in both sets. Class sizes range from 336 (Healthy) down to 11 (Leukemia with thrombocytopenia).
- Random Forest with balanced class weights to compensate for the imbalance.
- Gradio demo: enter 14 comma-separated CBC values and get the diagnosis name back.

## Results
Train accuracy 98.0% vs. test accuracy 97.8%, so there is no sign of overfitting.

| Class | Test samples | Precision | Recall |
|---|---:|---:|---:|
| Healthy | 84 | 1.00 | 1.00 |
| Normocytic hypochromic anemia | 70 | 1.00 | 0.96 |
| Normocytic normochromic anemia | 67 | 1.00 | 0.97 |
| Iron deficiency anemia | 47 | 0.98 | 1.00 |
| Thrombocytopenia | 18 | 1.00 | 1.00 |
| Other microcytic anemia | 15 | 1.00 | 0.87 |
| Leukemia | 12 | 0.86 | 1.00 |
| Macrocytic anemia | 5 | 0.62 | 1.00 |
| Leukemia with thrombocytopenia | 3 | 0.75 | 1.00 |

## Limitations & next steps
- Educational project only, **not a diagnostic tool**.
- The rarest classes have only 3-5 test samples, so their scores are unstable. Cross-validation would give a more reliable estimate.
- The source data has suspicious values (e.g. HGB 0.4 with HCT 316 in row 4), and the quartiles of `LYMp`/`LYMn`/`NEUTp`/`NEUTn` all equal the column mean, which suggests mean-imputation in the original dataset. None of this is cleaned here.
- Worth checking how the labels were produced: if they derive from CBC thresholds, high accuracy is partly expected.
- Next: feature-importance analysis, outlier handling, and deploying the demo to Hugging Face Spaces.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix



In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ehababoelnaga/anemia-types-classification")

print("Path to dataset files:", path)

100%|██████████| 21.7k/21.7k [00:00<00:00, 20.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ehababoelnaga/anemia-types-classification/versions/1


In [3]:
df = pd.read_csv(path + "/diagnosed_cbc_data_v4.csv")
df

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT,Diagnosis
0,10.00,43.200,50.100,4.30000,5.00000,2.77,7.3,24.2000,87.7,26.3,30.1,189.0,12.500000,0.17000,Normocytic hypochromic anemia
1,10.00,42.400,52.300,4.20000,5.30000,2.84,7.3,25.0000,88.2,25.7,20.2,180.0,12.500000,0.16000,Normocytic hypochromic anemia
2,7.20,30.700,60.700,2.20000,4.40000,3.97,9.0,30.5000,77.0,22.6,29.5,148.0,14.300000,0.14000,Iron deficiency anemia
3,6.00,30.200,63.500,1.80000,3.80000,4.22,3.8,32.8000,77.9,23.2,29.8,143.0,11.300000,0.12000,Iron deficiency anemia
4,4.20,39.100,53.700,1.60000,2.30000,3.93,0.4,316.0000,80.6,23.9,29.7,236.0,12.800000,0.22000,Normocytic hypochromic anemia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,4.40,25.845,77.511,1.88076,5.14094,4.86,13.5,46.1526,80.7,27.7,34.4,180.0,14.312512,0.26028,Healthy
1277,5.60,25.845,77.511,1.88076,5.14094,4.85,15.0,46.1526,91.7,31.0,33.8,215.0,14.312512,0.26028,Healthy
1278,9.20,25.845,77.511,1.88076,5.14094,4.47,13.1,46.1526,88.7,29.3,33.0,329.0,14.312512,0.26028,Healthy
1279,6.48,25.845,77.511,1.88076,5.14094,4.75,13.2,46.1526,86.7,27.9,32.1,174.0,14.312512,0.26028,Healthy


In [4]:
df['Diagnosis'].value_counts()

,count
Diagnosis,
Healthy,336
Normocytic hypochromic anemia,279
Normocytic normochromic anemia,269
Iron deficiency anemia,189
Thrombocytopenia,73
Other microcytic anemia,59
Leukemia,47
Macrocytic anemia,18
Leukemia with thrombocytopenia,11


In [5]:
df.info(

)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1281 entries, 0 to 1280
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   WBC        1281 non-null   float64
 1   LYMp       1281 non-null   float64
 2   NEUTp      1281 non-null   float64
 3   LYMn       1281 non-null   float64
 4   NEUTn      1281 non-null   float64
 5   RBC        1281 non-null   float64
 6   HGB        1281 non-null   float64
 7   HCT        1281 non-null   float64
 8   MCV        1281 non-null   float64
 9   MCH        1281 non-null   float64
 10  MCHC       1281 non-null   float64
 11  PLT        1281 non-null   float64
 12  PDW        1281 non-null   float64
 13  PCT        1281 non-null   float64
 14  Diagnosis  1281 non-null   object 
dtypes: float64(14), object(1)
memory usage: 150.2+ KB


In [6]:
df['Diagnosis'] = df['Diagnosis'].map({'Healthy':0, 'Normocytic hypochromic anemia':1, 'Normocytic normochromic anemia':2, 'Iron deficiency anemia':3,'Thrombocytopenia':4,'Other microcytic anemia':5,'Leukemia':6,'Macrocytic anemia':7,'Leukemia with thrombocytopenia':8})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1281 entries, 0 to 1280
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   WBC        1281 non-null   float64
 1   LYMp       1281 non-null   float64
 2   NEUTp      1281 non-null   float64
 3   LYMn       1281 non-null   float64
 4   NEUTn      1281 non-null   float64
 5   RBC        1281 non-null   float64
 6   HGB        1281 non-null   float64
 7   HCT        1281 non-null   float64
 8   MCV        1281 non-null   float64
 9   MCH        1281 non-null   float64
 10  MCHC       1281 non-null   float64
 11  PLT        1281 non-null   float64
 12  PDW        1281 non-null   float64
 13  PCT        1281 non-null   float64
 14  Diagnosis  1281 non-null   int64  
dtypes: float64(14), int64(1)
memory usage: 150.2 KB


In [7]:
df['Diagnosis']

,Diagnosis
0,1
1,1
2,3
3,3
4,1
...,...
1276,0
1277,0
1278,0
1279,0


In [8]:
df['Diagnosis'].value_counts()

,count
Diagnosis,
0,336
1,279
2,269
3,189
4,73
5,59
6,47
7,18
8,11


In [9]:
df.describe()

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT,Diagnosis
count,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000,1281.0000,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000,1281.000000
mean,7.862717,25.845000,77.511000,1.880760,5.140940,4.708267,12.184551,46.1526,85.793919,32.084840,31.739149,229.981421,14.312512,0.260280,1.925839
std,3.564466,7.038728,147.746273,1.335689,2.872294,2.817200,3.812897,104.8861,27.177663,111.170756,3.300352,93.019336,3.005079,0.685351,1.817010
min,0.800000,6.200000,0.700000,0.200000,0.500000,1.360000,-10.000000,2.0000,-79.300000,10.900000,11.500000,10.000000,8.400000,0.010000,0.000000
25%,6.000000,25.845000,71.100000,1.880760,5.100000,4.190000,10.800000,39.2000,81.200000,25.500000,30.600000,157.000000,13.300000,0.170000,0.000000
50%,7.400000,25.845000,77.511000,1.880760,5.140940,4.600000,12.300000,46.1526,86.600000,27.800000,32.000000,213.000000,14.312512,0.260280,2.000000
75%,8.680000,25.845000,77.511000,1.880760,5.140940,5.100000,13.500000,46.1526,90.200000,29.600000,32.900000,293.000000,14.700000,0.260280,3.000000
max,45.700000,91.400000,5317.000000,41.800000,79.000000,90.800000,87.100000,3715.0000,990.000000,3117.000000,92.800000,660.000000,97.000000,13.600000,8.000000


In [10]:
x = df.drop('Diagnosis',axis=1)
y = df['Diagnosis']
#

In [11]:
x

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT
0,10.00,43.200,50.100,4.30000,5.00000,2.77,7.3,24.2000,87.7,26.3,30.1,189.0,12.500000,0.17000
1,10.00,42.400,52.300,4.20000,5.30000,2.84,7.3,25.0000,88.2,25.7,20.2,180.0,12.500000,0.16000
2,7.20,30.700,60.700,2.20000,4.40000,3.97,9.0,30.5000,77.0,22.6,29.5,148.0,14.300000,0.14000
3,6.00,30.200,63.500,1.80000,3.80000,4.22,3.8,32.8000,77.9,23.2,29.8,143.0,11.300000,0.12000
4,4.20,39.100,53.700,1.60000,2.30000,3.93,0.4,316.0000,80.6,23.9,29.7,236.0,12.800000,0.22000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,4.40,25.845,77.511,1.88076,5.14094,4.86,13.5,46.1526,80.7,27.7,34.4,180.0,14.312512,0.26028
1277,5.60,25.845,77.511,1.88076,5.14094,4.85,15.0,46.1526,91.7,31.0,33.8,215.0,14.312512,0.26028
1278,9.20,25.845,77.511,1.88076,5.14094,4.47,13.1,46.1526,88.7,29.3,33.0,329.0,14.312512,0.26028
1279,6.48,25.845,77.511,1.88076,5.14094,4.75,13.2,46.1526,86.7,27.9,32.1,174.0,14.312512,0.26028


In [24]:
y

,Diagnosis
0,1
1,1
2,3
3,3
4,1
...,...
1276,0
1277,0
1278,0
1279,0


In [12]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=13,stratify=y)

In [13]:
model = RandomForestClassifier(n_estimators=100,class_weight='balanced',ccp_alpha=0.02)
model.fit(x_train,y_train)

RandomForestClassifier(ccp_alpha=0.02, class_weight='balanced')

In [14]:
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

In [15]:


print('accuracy for train',accuracy_score(y_train,y_pred_train))
print('classification for train',classification_report(y_train,y_pred_train))
print('confusion for train',confusion_matrix(y_train,y_pred_train))

accuracy for train 0.9802083333333333
classification for train               precision    recall  f1-score   support

           0       1.00      1.00      1.00       252
           1       0.99      0.97      0.98       209
           2       0.98      0.98      0.98       202
           3       0.98      1.00      0.99       142
           4       0.96      1.00      0.98        55
           5       1.00      0.86      0.93        44
           6       1.00      1.00      1.00        35
           7       0.93      1.00      0.96        13
           8       0.62      1.00      0.76         8

    accuracy                           0.98       960
   macro avg       0.94      0.98      0.95       960
weighted avg       0.98      0.98      0.98       960

confusion for train [[251   0   0   0   0   0   0   1   0]
 [  1 202   4   0   0   0   0   0   2]
 [  0   3 197   0   0   0   0   0   2]
 [  0   0   0 142   0   0   0   0   0]
 [  0   0   0   0  55   0   0   0   0]
 [  0   0   0   3

In [16]:

print('accuracy for test',accuracy_score(y_test,y_pred_test))
print('classification for test',classification_report(y_test,y_pred_test))
print('confusion for test',confusion_matrix(y_test,y_pred_test))

accuracy for test 0.9781931464174455
classification for test               precision    recall  f1-score   support

           0       1.00      1.00      1.00        84
           1       1.00      0.96      0.98        70
           2       1.00      0.97      0.98        67
           3       0.98      1.00      0.99        47
           4       1.00      1.00      1.00        18
           5       1.00      0.87      0.93        15
           6       0.86      1.00      0.92        12
           7       0.62      1.00      0.77         5
           8       0.75      1.00      0.86         3

    accuracy                           0.98       321
   macro avg       0.91      0.98      0.94       321
weighted avg       0.98      0.98      0.98       321

confusion for test [[84  0  0  0  0  0  0  0  0]
 [ 0 67  0  0  0  0  1  2  0]
 [ 0  0 65  0  0  0  0  1  1]
 [ 0  0  0 47  0  0  0  0  0]
 [ 0  0  0  0 18  0  0  0  0]
 [ 0  0  0  1  0 13  1  0  0]
 [ 0  0  0  0  0  0 12  0  0]
 [ 0 

In [17]:
!pip install gradio

In [20]:
import gradio as gr

In [21]:
def pridect_type (numbers) :
  numbers = [numbers]
  y_pred_test = model.predict(numbers)
  return y_pred_test



In [22]:
demo = gr.Interface(fn=pridect_type,inputs='number',outputs='number')
demo.launch()

### Updated Gradio Demo with All 14 CBC Features

Since our model was trained on 14 features (`WBC`, `LYMp`, `NEUTp`, `LYMn`, `NEUTn`, `RBC`, `HGB`, `HCT`, `MCV`, `MCH`, `MCHC`, `PLT`, `PDW`, `PCT`), we need to provide these exact 14 values to make a valid prediction. We can map the numeric output back to our human-readable diagnosis labels.

In [23]:
import numpy as np
import gradio as gr

# Mapping from numeric predictions back to text labels
diagnosis_mapping = {
    0: 'Healthy',
    1: 'Normocytic hypochromic anemia',
    2: 'Normocytic normochromic anemia',
    3: 'Iron deficiency anemia',
    4: 'Thrombocytopenia',
    5: 'Other microcytic anemia',
    6: 'Leukemia',
    7: 'Macrocytic anemia',
    8: 'Leukemia with thrombocytopenia'
}

def predict_anemia_from_list(comma_separated_features):
    try:
        # Convert the comma-separated string input into a list of floats
        feature_list = [float(x.strip()) for x in comma_separated_features.split(',')]

        if len(feature_list) != 14:
            return f"Error: Please provide exactly 14 values. You provided {len(feature_list)}."

        # Reshape for a single sample prediction
        features_array = np.array(feature_list).reshape(1, -1)

        # Predict
        prediction_code = model.predict(features_array)[0]

        # Return translated label
        return diagnosis_mapping.get(prediction_code, f"Unknown Code: {prediction_code}")
    except Exception as e:
        return f"Error processing input: {str(e)}"

# Set up the updated interface
input_placeholder = "10.0, 43.2, 50.1, 4.3, 5.0, 2.77, 7.3, 24.2, 87.7, 26.3, 30.1, 189.0, 12.5, 0.17"

demo_list = gr.Interface(
    fn=predict_anemia_from_list,
    inputs=gr.Textbox(
        label="Enter 14 CBC values (comma-separated)",
        placeholder=input_placeholder,
        value=input_placeholder
    ),
    outputs=gr.Textbox(label="Predicted Diagnosis"),
    title="Anemia & CBC Diagnosis Classifier",
    description="Provide values for: WBC, LYMp, NEUTp, LYMn, NEUTn, RBC, HGB, HCT, MCV, MCH, MCHC, PLT, PDW, PCT (in order, separated by commas)."
)

demo_list.launch(share=True)